# `n_color_immerse()`

This document explains why assigning colors to three-dimensional nematic directors is a surprisingly nontrivial visualization problem. At first glance, a 3D director has three components and an RGB color also has three components, so one might expect a simple one-to-one coloring rule. However, the nematic symmetry $\mathbf n\sim-\mathbf n$ changes the topology of the orientation space and makes the most natural requirements on such a colormap mutually incompatible.

We will first explain where this obstruction comes from, why an ordinary one-to-one continuous coloring is mathematically impossible, and why an immersion provides the appropriate compromise.

## Background

A three-dimensional nematic director is represented by a unit vector $\mathbf n\in S^2$, but the two vectors $\mathbf n$ and $-\mathbf n$ describe the same physical orientation. The space of distinct nematic orientations is therefore not the sphere itself, but the real projective plane

$$
\mathbb{RP}^2=S^2/(\mathbf n\sim-\mathbf n).
$$

A director colormap can thus be viewed as a map from $\mathbb{RP}^2$ into RGB color space. If we ignore the finite bounds of RGB for the moment, color space is locally just three-dimensional Euclidean space $\mathbb R^3$. The most desirable colormap would be continuous and one-to-one: nearby orientations would have nearby colors, while every distinct orientation would have a distinct color. Topologically, this would amount to an **embedding** of $\mathbb{RP}^2$ in $\mathbb R^3$.

Such an embedding does not exist. The real projective plane is a closed non-orientable surface, and it cannot be embedded in three-dimensional Euclidean space without self-intersection. Consequently, no continuous RGB colormap can simultaneously preserve the nematic identification and assign a globally unique color to every director orientation. This is a topological obstruction rather than a limitation of a particular coloring formula.

The obstruction becomes much less restrictive if we replace an embedding by an **immersion**. An immersion is locally regular: around every point, the surface is mapped smoothly into $\mathbb R^3$ without collapsing a local direction. Unlike an embedding, however, an immersion is allowed to intersect itself globally. Thus two distinct points of $\mathbb{RP}^2$ may occasionally be mapped to the same point in $\mathbb R^3$, while the map still preserves the local two-dimensional structure of orientation space.

This distinction is exactly what we need for director coloring. Since a perfect one-to-one continuous map is impossible, we instead seek an immersion whose self-intersections are as controlled as possible and whose geometry makes different orientations as distinguishable as possible after conversion to RGB colors.

A classical realization of this idea is **Boy's surface**, an immersion of $\mathbb{RP}^2$ into $\mathbb R^3$. Boy's surface provides a concrete geometric model of the nematic orientation space in three dimensions: the unavoidable topological conflict is concentrated into self-intersections rather than resolved by introducing a discontinuity or an arbitrary sign convention for $\mathbf n$. This makes it a natural starting point for constructing a three-dimensional nematic colormap.

There is not, however, a unique Boy's surface. Different immersions and different geometric realizations distribute distortion and self-intersection differently. For visualization, these differences matter: after the immersed coordinates are transformed into a realizable color space, they determine how well separated the resulting colors are and how uniformly orientation changes are represented. The next problem is therefore not merely to find *a* Boy's surface, but to find a realization that is particularly suitable for coloring nematic directors.

## Design goals

Our goal is now to find an **optimal Boy's surface** for director coloring. Rather than searching over arbitrary immersions of $\mathbb{RP}^2$, we start from a fixed Boy's-surface immersion and apply an invertible affine transformation in $\mathbb R^3$. If the original immersion is written as

$$
\mathbf f(\mathbf n):\mathbb{RP}^2\rightarrow\mathbb R^3,
$$

we consider the family

$$
\mathbf f_{A,\mathbf b}(\mathbf n)=A\,\mathbf f(\mathbf n)+\mathbf b,
$$

where $A$ is an invertible $3\times3$ matrix and $\mathbf b$ is a translation vector. Because an invertible affine map is a diffeomorphism of $\mathbb R^3$, this transformation does not change the topology of the immersed projective plane: it preserves the nematic identification, does not introduce singularities into the immersion, and does not alter the basic topological structure of its self-intersections. It only changes the geometry and placement of the realization in three-dimensional space.

This gives us a finite-dimensional optimization problem. The entries of $A$ together with the components of $\mathbf b$ are treated as adjustable parameters, and we search for the affine transformation that maximizes a utility function measuring how suitable the resulting surface is for visualization. In other words, instead of asking which Boy's surface is mathematically preferable in the abstract, we ask which affine deformation of a valid Boy's surface best serves the practical purpose of distinguishing nematic orientations by color.

The remaining question is therefore how to define *best*. Several properties matter for a useful director colormap, and they need not favor the same transformation. We introduce these requirements separately below.

### Locally uniform color variation

The first requirement is **local uniformity**. Equal changes of director orientation should produce comparable perceptual color changes wherever they occur on $\mathbb{RP}^2$. Otherwise, some regions of orientation space would be visually exaggerated while others would be compressed. We therefore measure the **local tangent-metric distortion** in OKLab. This quantity, denoted $J_{\mathrm{loc}}$, compares the local color-space metric induced by the colormap with the natural local metric of director orientation space. Smaller $J_{\mathrm{loc}}$ means that nearby orientations are represented more uniformly.

### Recognizable Cartesian axes

For practical visualization, the three Cartesian director axes should also have immediately recognizable colors. We associate the $x$, $y$, and $z$ directors with red, green, and blue, respectively. This is not required by the topology, but it makes plots substantially easier to read: a user can identify the dominant orientation without first learning an arbitrary color convention. We quantify this requirement by the perceptual OKLab distances between the colors assigned to the three axis directors and their target red, green, and blue colors. The allowed axis-color errors are calibrated against the original Nematics3D map, so improving vividness is not allowed to come at the cost of making these familiar reference directions less recognizable than the established coloring.

### Vivid colors

After local uniformity and the Cartesian reference colors have been controlled, we would like the remaining map to be as vivid as possible. A mathematically valid immersion could occupy only a small, nearly gray region of color space, but such a map would make different orientations unnecessarily difficult to distinguish. We therefore reward **chroma**, measured in OKLab, and use the mean chroma over director space as the vividness utility. Increasing this utility expands the useful color variation of the map rather than wasting the available color space near the gray axis.

There is a genuine tradeoff between vividness and local uniformity: allowing more local distortion generally permits a more colorful map. We therefore treat these quantities through a Pareto analysis rather than pretending that one arbitrary weighted sum defines the answer. The selected map is taken from the resulting chroma--$J_{\mathrm{loc}}$ Pareto frontier while respecting the axis-color requirements.

### The sRGB gamut: a hard constraint

Finally, every color produced by the map must actually be displayable. The affine transformation acts in three-dimensional coordinates that are ultimately interpreted as encoded sRGB values, so the complete immersed surface must remain inside the sRGB cube,

$$
0\le R,G,B\le1.
$$

This is fundamentally different from the utilities above. Leaving the sRGB gamut is not merely undesirable; it produces an invalid display color and would require clipping, which would deform the optimized map and can collapse distinct colors onto the gamut boundary. We therefore impose gamut containment as a **hard constraint** on the optimization rather than assigning it a penalty in the utility function.

Taken together, the optimization seeks as much perceptual chroma as possible while controlling local distortion and preserving recognizable axis colors, subject always to the requirement that the entire Boy's-surface image remain inside the sRGB gamut.

## Implementation

## Usage

## Visualizing the full colormap: the color sphere